# Human influence in Rural Wildfires ignitions

## Group: 02
### Name(s) & ID(s) of Group Members:



## Table of Contents

1. [Research Question](#1-research-question)
2. [Collect Data](#2-collect-data)
3. [Explore](#3-explore)
4. [Draw Inferences](#4-draw-inferences)
5. [Formulate Conclusions](#5-formulate-conclusions)
6. [Look Back and Ahead](#6-look-back-and-ahead)

---

## 1) Research Question

How did initial response times and burnt areas change before and after 2017, and how are these two variables associated across four Portuguese municipalities?

### Secondary Research Questions:

1. Was the change in the proportion of negligent causes concentrated in the critical summer period, or did it also affect the traditional burning months (autumn/winter)?

    Difference in proportion (Variables to use: cause_type, period, season_risk_period). Test: Z test

2. Did the change in the proportion of negligent ignitions translate into a reduction in ignitions on weekends compared to weekdays?

    Median comparison (Variables to use: Daily_ignitions_count, day_type). Test: t test

3. Did the disparity in final burnt area between intentional and negligent causes narrow in the post-2017 period in the municipalities under study?

    ANOVA (Variables to use: Total_Burnt_Area, Cause_Type_Period). Test: ANOVA two-way 

We would need a 4th question with only a one way ANOVA

---

## 2) Collect data

### Dataset Source

The dataset used in this project was sourced from the Institute for Nature Conservation and Forests (ICNF), specifically the wildfire statistics published under the Information Management section of [their website](https://www.icnf.pt/florestas/gfr/gfrgestaoinformacao/estatisticas). This dataset contains data on human influenced wildfire ignitions from 2001 until 2025. The records are split across three ICNF files (2001-2010, 2011-2020 and 2021-2025), and were combined into a single dataset before analysis.

The raw file contains 40 columns covering fire identifiers, location, timestamps, burnt area by land cover, fire weather indices, and cause classification. Only a subset of these columns was used in the analysis, and several variables were derived from them. The full mapping is described below so that the variables can be reproduced from the raw file.

### Dataset Details


#### Variables: 

##### Qualitative Variables

- **Cause_Type**: Intentional vs. Negligent. Derived from the cause classification columns (`TipoCausa`, `GrupoCausa`, `CodCausa`), keeping only validated human causes. Fires with an unclassified or unknown cause were excluded.

- **Period**: Pre-2017 (2001–2016) vs. Post-2017 (2018–2025). Derived from `Ano`. In practice, the law was only implemented in 2021, so the post-2017 group mixes pre-law and post-law years (see limitations).

- **Municipality**: Chaves, Montalegre, Ourém, Tomar. Taken from `Concelho`.

- **Season_Risk_Period**: Critical Period vs. Non-Critical Period. Derived from `Mes` and `Dia`, with the critical period defined as 1 July to 30 September.

- **Day_Type**: Weekend vs. Weekday. Derived from the fire date (`Ano`, `Mes`, `Dia`), with Saturday and Sunday classified as weekend.

- **Cause_Type_Period**: Combined factor of `Cause_Type` and `Period`, used for the two-way ANOVA in Q3.

##### Quantitative Variables

- **Daily_Ignitions_Count**: Number of ignitions per municipality per day, computed by counting `Codigo_SGIF` grouped by `date` and `municipality`.

- **Total_Burnt_Area**: Total surface burnt per occurrence, in hectares, from `AreaTotal_ha`.

- **Initial_Response_Time**: Time in minutes between `DataHoraAlerta` and `DataHora_PrimeiraIntervencao`.

- **Fire_Duration**: Total duration from fire alert to final extinction, in hours, from `Duracao_Horas`.

#### Missing values

| Question | Variables used | rows dropped where |
|---|---|---|
| Q1 | `Cause_Type`, `Period`, `Season_Risk_Period` | `Cause_Type` is missing |
| Q2 | `Daily_Ignitions_Count`, `Day_Type` | date is missing (so the count cannot be built) |
| Q3 | `Total_Burnt_Area`, `Cause_Type_Period` | `Cause_Type` or `AreaTotal_ha` is missing |
| Q4 | `Total_Burnt_Area`, `Municipality` | `AreaTotal_ha` is missing |

### Data preparation

In [13]:
import numpy as np
import pandas as pd
from pathlib import Path

project_path = Path.cwd()
raw = project_path / "01_dataset/raw"
processed = project_path / "01_dataset/processed"
processed.mkdir(parents=True, exist_ok=True)

In [14]:
# Loading data
df1 = pd.read_excel(raw / "Registos_Incendios_SGIF_2021_2025.xlsx")
df2 = pd.read_excel(raw / "Registos_Incendios_SGIF_2011_2020.xlsx")

# confirm the raw files share the same schema
df1.columns.equals(df2.columns)

Before deriving variables, the sample size was checked by municipality and year to confirm that all four municipalities are represented across the full period.

In [ ]:
df = pd.concat([df1, df2], ignore_index=True)

concelhos = ["Chaves", "Ourém", "Montalegre", "Tomar"]

df_final = df[df["Concelho"].isin(concelhos)].copy()
df_final = df_final[df_final["Ano"].between(2011, 2025)].copy()

print(df_final["Concelho"].value_counts())
print(pd.crosstab(df_final["Concelho"], df_final["Ano"], margins=True))

Montalegre    3143
Chaves        1836
Ourém         1175
Tomar         1101
Name: Concelho, dtype: int64
Ano         2011  2012  2013  2014  2015  2016  2017  2018  2019  2020  2021  \
Concelho                                                                       
Chaves       321   292   191    58   181   106   132    67    66    73    46   
Montalegre   484   255   239    92   321   292   314   119   168   134   191   
Ourém         83   120    57    45    78    76    96    82    63    97    64   
Tomar         60    79   104    79   111    88   162    95    89    51    56   
All          948   746   591   274   691   562   704   363   386   355   357   

Ano         2022  2023  2024  2025   All  
Concelho                                  
Chaves       105    36    68    94  1836  
Montalegre   192   117    94   131  3143  
Ourém         87    70    83    74  1175  
Tomar         49    33    24    21  1101  
All          433   256   269   320  7255  


In [ ]:
#quantitative variables:

#Initial_Response_Time
for col in ["DataHoraAlerta", "DataHora_PrimeiraIntervencao", "DataHora_Extincao"]:
    df_final[col] = pd.to_datetime(df_final[col], errors="coerce")

df_final["Initial_Response_Time"] = (
    df_final["DataHora_PrimeiraIntervencao"] - df_final["DataHoraAlerta"]
).dt.total_seconds() / 60

#Fire_Duration
df_final["Fire_Duration"] = df_final["Duracao_Horas"]

#Total_Burnt_Area
df_final["Total_Burnt_Area"] = df_final["AreaTotal_ha"]

In [ ]:
#qualitative variables:

#Period
df_final["Period"] = np.where(df_final["Ano"] <= 2016, "Pre-2017",
                       np.where(df_final["Ano"] >= 2018, "Post-2017", "Transition"))

#Season_Risk_Period
df_final["Season_Risk_Period"] = np.where(
    df_final["Mes"].isin([7, 8, 9]),
    "Critical", "Non-Critical"
)

#Day_Type
df_final["Day_Type"] = np.where(
    df_final["DataHoraAlerta"].dt.dayofweek >= 5,
    "Weekend", "Weekday"
)

In [ ]:
#Ignition count

df_final["Date"] = df_final["DataHoraAlerta"].dt.date

daily = (
    df_final.groupby(["Concelho", "Date"])
            .size()
            .reset_index(name="Daily_Ignitions_Count")
)

daily["Day_Type"] = np.where(
    pd.to_datetime(daily["Date"]).dt.dayofweek >= 5,
    "Weekend", "Weekday"
)

In [ ]:
#keep only validated human causes
cause_map = {
    "Negligente": "Negligent",
    "Intencional": "Intentional",
}

df_final["Cause_Type"] = df_final["TipoCausa"].map(cause_map)

df_final["Cause_Type_Period"] = np.where(
    df_final["Cause_Type"].isna(),
    np.nan,
    df_final["Cause_Type"].astype(str) + "_" + df_final["Period"].astype(str)
)

In [ ]:
df_final.describe

<bound method NDFrame.describe of        Codigo_SGIF   Codigo_ANEPC   Ano  Mes  Dia  Hora  AreaPov_ha  \
6           TM1216  2021170000883  2021    1    8    18    0.000000   
27         TM12115  2021170001075  2021    1   10    17    0.069071   
29         TM12116  2021170001076  2021    1   10    17    0.000000   
53         TM12118  2021170001335  2021    1   12    23    0.000000   
57         TM12120  2021170001399  2021    1   13    13    0.000000   
...            ...            ...   ...  ...  ...   ...         ...   
176941    TM120886  2020170028056  2020   10   19    12    0.000000   
176942    TM120885  2020170028057  2020   10   19    12    0.000000   
176946    TM120883  2020170028068  2020   10   19    13    0.000000   
176956    TM120879  2020170028086  2020   10   19    15    0.000000   
177123    TM120924  2020170033972  2020   12   26    13    0.000000   

        AreaMato_ha  AreaAgric_ha  AreaTotal_ha  ...          DC        DMC  \
6          0.050000           0.0 

Saving processed data to be used for statistics

In [ ]:
df_final.to_excel(processed / "wildfires_processed.xlsx", index=False)
daily.to_excel(processed / "daily_ignitions.xlsx", index=False)

Per question dataset

In [ ]:
# Q1: negligent vs intentional, pre vs post, critical vs non-critical
q1 = df_final.dropna(subset=["Cause_Type"]).copy()
q1.to_excel(processed / "q1_cause_period_season.xlsx", index=False)

# Q2: daily ignitions, weekend vs weekday
q2 = daily.dropna(subset=["Date", "Daily_Ignitions_Count"]).copy()
q2.to_excel(processed / "q2_daily_ignitions.xlsx", index=False)

# Q3: burnt area by cause type and period (two-way ANOVA)
q3 = df_final.dropna(subset=["Cause_Type", "Total_Burnt_Area"]).copy()
q3.to_excel(processed / "q3_burnt_area_cause_period.xlsx", index=False)

# Q4: burnt area by municipality (one-way ANOVA)
q4 = df_final.dropna(subset=["Total_Burnt_Area", "Concelho"]).copy()
q4.to_excel(processed / "q4_burnt_area_municipality.xlsx", index=False)

KeyError: ['Cause_Type']

## 6) Look back and ahead

### Limitations of the Dataset

Two limitations affect interpretation and should be kept in mind when reading the results.

First, the Period split is set at 2017, but the law was only implemented in 2021. The post-2017 group therefore mixes years before and after the law took effect, so any change attributed to the post-2017 period cannot be separated from the effect of the law itself without a further split.

Second, Initial_Response_Time and Fire_Duration depend on how consistently timestamps are recorded across municipalities. The alert source (FonteAlerta) also varies, and response time depends partly on who reported the fire. These differences may introduce variation between municipalities that is not caused by fire behaviour, which is relevant to the municipality comparison in Q4.